# 02 -- Feature Engineering

**Purpose:** Clean, transform, and engineer new features for better model performance.

| Step | Description |
|---|---|
| 1 | Load raw data |
| 2 | Parse date features |
| 3 | Encode categorical variables |
| 4 | Engineer interaction features |
| 5 | Outlier analysis (IQR) |
| 6 | Log-transform target |
| 7 | Scale features |
| 8 | Save processed data |

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src.config import (RAW_DATA_PATH, PROCESSED_DATA_PATH, IMAGES_DIR,
                        NUMERICAL_COLS, CATEGORICAL_COLS, TARGET_COL, TARGET_LOG_COL)
from src.data_loader import load_raw_data

pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [2]:
df = load_raw_data(RAW_DATA_PATH)
print(df.shape)
df.head(3)

Loaded 9,134 rows x 24 cols from AutoInsurance.csv
(9134, 24)


,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,Income,Location Code,Marital Status,Monthly Premium Auto,Months Since Last Claim,Months Since Policy Inception,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size
0,BU79786,Washington,2763.519279,No,Basic,Bachelor,2/24/11,Employed,F,56274,Suburban,Married,69,32,5,0,1,Corporate Auto,Corporate L3,Offer1,Agent,384.811147,Two-Door Car,Medsize
1,QZ44356,Arizona,6979.535903,No,Extended,Bachelor,1/31/11,Unemployed,F,0,Suburban,Single,94,13,42,0,8,Personal Auto,Personal L3,Offer3,Agent,1131.464935,Four-Door Car,Medsize
2,AI49188,Nevada,12887.431650,No,Premium,Bachelor,2/19/11,Employed,F,48767,Suburban,Married,108,18,38,0,2,Personal Auto,Personal L3,Offer1,Agent,566.472247,Two-Door Car,Medsize


## 2. Parse Date Features

In [3]:
# Extract month from Effective To Date -- seasonality signal
df['Effective To Date'] = pd.to_datetime(df['Effective To Date'], errors='coerce')
df['Effective_Month'] = df['Effective To Date'].dt.month
df = df.drop(columns=['Effective To Date'])
print("Extracted Effective_Month. Value counts:")
print(df['Effective_Month'].value_counts().sort_index())

Extracted Effective_Month. Value counts:
Effective_Month
1     3356
2     2722
3      336
4      279
5      325
6      287
7      308
8      283
9      283
10     356
11     317
12     282
Name: count, dtype: int64


## 3. Outlier Analysis (IQR)

In [4]:
num_cols = [c for c in NUMERICAL_COLS if c in df.columns]
outlier_summary = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({'Feature': col, 'Q1': Q1, 'Q3': Q3,
                            'IQR': IQR, 'Lower': lower, 'Upper': upper,
                            'Outliers': n_out, 'Pct': round(100*n_out/len(df), 2)})

out_df = pd.DataFrame(outlier_summary).set_index('Feature')
print(out_df[['Outliers', 'Pct', 'Lower', 'Upper']])
print("\nDecision: KEEP all outliers -- insurance CLV naturally has extreme values.")
print("Capping would destroy legitimate high-value customer signals.")

                               Outliers    Pct         Lower         Upper
Feature                                                                   
Income                                0   0.00 -93480.000000  155800.00000
Monthly Premium Auto                430   4.71      6.500000     170.50000
Months Since Last Claim               0   0.00    -19.500000      48.50000
Months Since Policy Inception         0   0.00    -46.500000     141.50000
Number of Open Complaints          1882  20.60      0.000000       0.00000
Number of Policies                  416   4.55     -3.500000       8.50000
Total Claim Amount                  453   4.96   -140.626647     960.39973

Decision: KEEP all outliers -- insurance CLV naturally has extreme values.
Capping would destroy legitimate high-value customer signals.


## 4. Engineer Interaction Features

In [5]:
# Log-transform target
df[TARGET_LOG_COL] = np.log1p(df[TARGET_COL])

# Interaction: total premium commitment
df['Premium_x_Policies'] = df['Monthly Premium Auto'] * df['Number of Policies']

# Temporal gap: policy age vs last claim
df['Policy_Claim_Gap'] = (df['Months Since Policy Inception'] -
                          df['Months Since Last Claim'])

# Claims efficiency per premium dollar
df['Claim_to_Premium_Ratio'] = (df['Total Claim Amount'] /
                                 (df['Monthly Premium Auto'] + 1))

# Income per policy
df['Income_per_Policy'] = df['Income'] / (df['Number of Policies'] + 1)

print("Engineered features added:")
new_feats = [TARGET_LOG_COL, 'Premium_x_Policies', 'Policy_Claim_Gap',
             'Claim_to_Premium_Ratio', 'Income_per_Policy']
print(df[new_feats].describe().round(3))

Engineered features added:
        CLV_log  Premium_x_Policies  Policy_Claim_Gap  Claim_to_Premium_Ratio  \
count  9134.000            9134.000          9134.000                9134.000   
mean      8.749             275.581            32.968                   4.549   
std       0.653             257.724            30.073                   2.252   
min       7.549              61.000           -34.000                   0.002   
25%       8.293             105.000             8.000                   3.238   
50%       8.662             188.000            33.000                   4.736   
75%       9.101             357.000            57.000                   5.530   
max      11.331            2664.000            99.000                  12.055   

       Income_per_Policy  
count           9134.000  
mean           12503.522  
std            12230.576  
min                0.000  
25%                0.000  
50%             9549.667  
75%            19578.750  
max            49990.500  


In [6]:
# Plot engineered features vs log CLV
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
eng_feats = ['Premium_x_Policies', 'Policy_Claim_Gap',
             'Claim_to_Premium_Ratio', 'Income_per_Policy']
for i, feat in enumerate(eng_feats):
    axes[i].scatter(df[feat], df[TARGET_LOG_COL], alpha=0.3, s=8, color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('log(CLV)')
    axes[i].set_title(f'{feat} vs log(CLV)')
plt.suptitle('Engineered Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'engineered_features_vs_clv.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: engineered_features_vs_clv.png")

Saved: engineered_features_vs_clv.png


## 5. Encode Categorical Features

In [7]:
# Drop Customer ID
df = df.drop(columns=['Customer'], errors='ignore')

cat_cols = [c for c in CATEGORICAL_COLS if c in df.columns]
print(f"Encoding {len(cat_cols)} categorical columns: {cat_cols}")

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"\nShape after encoding: {df.shape}")

Encoding 14 categorical columns: ['State', 'Response', 'Coverage', 'Education', 'EmploymentStatus', 'Gender', 'Location Code', 'Marital Status', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Vehicle Class', 'Vehicle Size']

Shape after encoding: (9134, 57)


## 6. Final Feature Matrix Inspection

In [8]:
feature_cols = [c for c in df.columns if c not in [TARGET_COL, TARGET_LOG_COL]]
print(f"Total features: {len(feature_cols)}")
print("\nFeature list (first 20):", feature_cols[:20])
print("\nMissing after engineering:", df.isnull().sum().sum())

Total features: 55

Feature list (first 20): ['Income', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Total Claim Amount', 'Effective_Month', 'Premium_x_Policies', 'Policy_Claim_Gap', 'Claim_to_Premium_Ratio', 'Income_per_Policy', 'State_California', 'State_Nevada', 'State_Oregon', 'State_Washington', 'Response_Yes', 'Coverage_Extended', 'Coverage_Premium', 'Education_College']

Missing after engineering: 0


## 7. Save Processed Data

In [9]:
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved processed data -> {PROCESSED_DATA_PATH}")
print(f"Shape: {df.shape}")

Saved processed data -> D:\automation\Customer-Lifetime-Value-Prediction-For-AutoInsurance-Company\data\processed\processed_data.csv
Shape: (9134, 57)


## Feature Engineering Summary

| Feature | Formula | Rationale |
|---|---|---|
| CLV_log | log1p(CLV) | Normalize right-skewed target for better linear model fit |
| Effective_Month | month(Effective To Date) | Seasonality -- policy effective month may correlate with renewal behavior |
| Premium_x_Policies | Monthly Premium x Num Policies | Total monthly premium commitment -- strong CLV signal |
| Policy_Claim_Gap | Months Inception - Months Since Claim | Loyal customer who hasn't claimed recently |
| Claim_to_Premium_Ratio | Total Claims / (Premium + 1) | Claims efficiency -- high ratio = risky customer |
| Income_per_Policy | Income / (Num Policies + 1) | Affordability normalized by portfolio size |

**Outlier Decision:** Kept all outliers — insurance CLV legitimately has high-value customers; capping would destroy the signal.

---
**Next step: `03_Feature_Selection.ipynb`**